In [4]:
"""
verify_spectral.py (versión final)

Verifica la matriz de error CORRECTA (I-LC)A, sintetiza la ganancia vía DARE,
calcula las constantes de Lyapunov, valida empíricamente τ_e sobre los logs
reales, e imprime bloques LaTeX listos para pegar en el paper y las cartas.

Compatible con Jupyter (%run verify_spectral.py) y consola
    python verify_spectral.py --csv-dir "C:\\Users\\PC\\Downloads\\articulo"
"""
import sys, os, re, glob, argparse
import numpy as np
import scipy.linalg as la
from numpy.linalg import eigvals, eigvalsh

# ============================================================================
# 1. MATRICES DEL PAPER
# ============================================================================
A = np.array([
    [ 0.7941,  0.1507,  0.0724,  0.0216,  0.0302,  0.0344,  0.0143, -0.0942],
    [ 0.1946,  0.7226,  0.0680,  0.0548,  0.0739, -0.1497,  0.0084,  0.0127],
    [ 0.0365,  0.0833,  0.8535,  0.0173, -0.0707, -0.0142, -0.0340,  0.0932],
    [-0.0516,  0.0113,  0.0399,  0.5721,  0.3169,  0.1792,  0.1078, -0.1236],
    [ 0.0516,  0.0042, -0.0619,  0.3049,  0.3937,  0.2134, -0.1147,  0.1401],
    [ 0.0352, -0.0175, -0.0259,  0.1833,  0.1875,  0.5980, -0.0418,  0.0639],
    [-0.0316,  0.0987, -0.0783,  0.2436, -0.0136, -0.2643,  0.8455,  0.1724],
    [-0.0984,  0.0471,  0.0607, -0.0872,  0.1991, -0.0421,  0.1493,  0.8004],
])
B = np.array([
    [ 0.0194, -0.0005],
    [ 0.0023,  0.0007],
    [ 0.0001,  0.0211],
    [ 0.0029,  0.0002],
    [ 0.0003,  0.0026],
    [ 0.0022,  0.0028],
    [ 0.0159, -0.0001],
    [ 0.0001,  0.0152],
])
L_paper = np.array([
    [ 0.5225,  0.0485,  0.0257, -0.0442],
    [ 0.2277,  0.1175,  0.1043,  0.0318],
    [ 0.0295,  0.5365, -0.0048,  0.0543],
    [ 0.1977, -0.1857,  0.0683,  0.0538],
    [ 0.2178, -0.2120, -0.0837,  0.1917],
    [ 0.2155, -0.2040, -0.0759,  0.1508],
    [-0.0023, -0.0319,  0.5867,  0.1208],
    [-0.0441,  0.0348,  0.1149,  0.4999],
])
C = np.zeros((4, 8))
C[0, 0] = C[1, 2] = C[2, 6] = C[3, 7] = 1.0

nz, Ts, EPS_PSI = 8, 2.0, 1e-8


# ============================================================================
# 2. UTILIDADES
# ============================================================================
def spectral_report(Ac, label):
    rho = np.abs(eigvals(Ac)).max()
    tau = -Ts / np.log(rho) if rho < 1.0 else float('inf')
    tag = "OK" if rho < 1.0 else "⚠ NO ESTABLE"
    print(f"  {label:<26} ρ = {rho:.6f}   τ_e = {tau:10.2f} s   {tag}")
    return rho, tau


def lyapunov_constants(Ac):
    W = la.solve_discrete_lyapunov(Ac.T, np.eye(nz))
    W = 0.5 * (W + W.T)
    assert np.all(eigvalsh(W) > 0), "W no es PD: A_c no es Schur"
    lam = eigvalsh(W)
    gamma = np.sqrt(1.0 - 1.0 / lam.max())
    kappa = lam.max() / lam.min()
    return dict(lam_max=lam.max(), lam_min=lam.min(), kappa=kappa,
                gamma=gamma, sqrt_kappa=np.sqrt(kappa),
                amplif=np.sqrt(kappa) / (1.0 - gamma))


def psi(x):
    h1, h2, h3 = x
    return np.array([np.sqrt(max(h1, EPS_PSI)), np.sqrt(max(h2, EPS_PSI)),
                     np.sqrt(max(h3, EPS_PSI)), h1 * h2, h2 * h3, h1 * h3,
                     h1, h3], dtype=float)


def run_num(path):
    m = re.search(r'run(\d+)', os.path.basename(path))
    return int(m.group(1)) if m else -1


# ============================================================================
# 3. P1 — Verificación espectral (I-LC)A  vs  A-LC
# ============================================================================
print("=" * 82)
print(" P1. VERIFICACIÓN ESPECTRAL — (I-LC)A vs A-LC")
print("=" * 82)
print(f"\n  |L|max  = {np.abs(L_paper).max():.4f}")
print(f"  ρ(A)    = {np.abs(eigvals(A)).max():.5f}")

A_c_correct = (np.eye(nz) - L_paper @ C) @ A
A_c_wrong   = A - L_paper @ C

print("\n  Matrices candidatas:")
rho_correct, tau_correct = spectral_report(A_c_correct, "(I-LC)A  [CORRECTA]")
rho_wrong,   tau_wrong   = spectral_report(A_c_wrong,   "A-LC     [INCORRECTA]")

ev_c = np.sort(np.abs(eigvals(A_c_correct)))[::-1]
ev_w = np.sort(np.abs(eigvals(A_c_wrong)))[::-1]
print("\n  Autovalores (módulo, ordenados):")
print("   (I-LC)A:", np.round(ev_c, 4))
print("   A-LC   :", np.round(ev_w, 4))
delta = np.abs(ev_c - ev_w).max()
print(f"   Δ máx entre espectros = {delta:.2e}  "
      f"{'→ valores viejos recuperables' if delta < 1e-2 else '→ reemplazar'}")


# ============================================================================
# 4. P2 — Reproducción del gain por DARE
# ============================================================================
print("\n" + "=" * 82)
print(" P2. REPRODUCCIÓN DEL GAIN VÍA DARE (alpha=1, R=I4)")
print("=" * 82)
X_dare = la.solve_discrete_are(A.T, C.T, np.eye(nz), np.eye(4))
M_inv  = np.linalg.inv(C @ X_dare @ C.T + np.eye(4))
L_pred = A @ X_dare @ C.T @ M_inv        # predictor form (con A)
L_filt =     X_dare @ C.T @ M_inv        # filter    form (sin A)

d_pred = np.abs(L_paper - L_pred).max()
d_filt = np.abs(L_paper - L_filt).max()
print(f"\n  ||L_paper - L_predictor||_max = {d_pred:.2e}")
print(f"  ||L_paper - L_filter   ||_max = {d_filt:.2e}")

if d_pred < 1e-3 and d_filt > 1e-2:
    print("  → El paper usa la forma PREDICTOR: L = A X Cᵀ (CXCᵀ+R)⁻¹")
elif d_filt < 1e-3 and d_pred > 1e-2:
    print("  → El paper usa la forma FILTER: L = X Cᵀ (CXCᵀ+R)⁻¹")
else:
    print("  → Ambigua: revisar manualmente")

# Ganancia de trabajo para el resto del script
if rho_correct < 1.0:
    print("\n✓ (I-LC)A ES Schur con el gain del paper.")
    print("  → Solo corregir notación + reemplazar ρ, τ_e, γ, κ(W).")
    L_use, A_c_use = L_paper, A_c_correct
    rho_use, tau_use = rho_correct, tau_correct
else:
    print("\n⚠ (I-LC)A NO es Schur con el gain del paper.")
    print("  → Usando la forma filter (L = X Cᵀ(CXCᵀ+R)⁻¹) como sustituto.")
    print("  → Requiere re-ejecutar replay con L_filt (offline, sin repetir experimentos).")
    L_use, A_c_use = L_filt, (np.eye(nz) - L_filt @ C) @ A
    rho_use, tau_use = spectral_report(A_c_use, "ρ((I-L_f C)A)")


# ============================================================================
# 5. P3 — Constantes de Lyapunov
# ============================================================================
print("\n" + "=" * 82)
print(" P3. CONSTANTES DE LYAPUNOV  (A_c = (I-LC)A)")
print("=" * 82)
const = lyapunov_constants(A_c_use)
print(f"\n  λ_max(W) = {const['lam_max']:10.4f}   (paper viejo: 11.32)")
print(f"  λ_min(W) = {const['lam_min']:10.4f}   (paper viejo: 1.001)")
print(f"  κ(W)     = {const['kappa']:10.4f}   (paper viejo: 11.31)")
print(f"  γ        = {const['gamma']:10.5f}   (paper viejo: 0.9548)")
print(f"  √κ(W)    = {const['sqrt_kappa']:10.4f}")
print(f"  factor amplificación √κ(W)/(1-γ) = {const['amplif']:.2f}")


# ============================================================================
# 6. P4 — Predictor puro (L=0)
# ============================================================================
print("\n" + "=" * 82)
print(" P4. PREDICTOR PURO (L=0) — sanity check")
print("=" * 82)
spectral_report(A, "ρ(A) = ρ((I-0·C)A)")
print("  → NO converge dentro de 800 s (τ_predictor ≫ 800 s).")


# ============================================================================
# P5 (corregido). Verificación empírica de τ_e por INYECCIÓN CONTROLADA
# ============================================================================
def replay_injected(path, A_use, B_use, L_use, C_use, delta0=0.02):
    """Replay con ĥ₂(0) = h₂_real(0) + delta0. Devuelve X (estados estimados)."""
    import pandas as pd
    df = pd.read_csv(path); df.columns = df.columns.str.strip()
    T   = df['t_s'].values
    h1  = df['h1_meas_m'].values
    h3  = df['h3_meas_m'].values
    h2r = df['h2_real_m'].values
    u   = np.column_stack([df['u1'].values, df['u3'].values])
    n = len(T)
    h2_0 = h2r[0] + delta0                      # <-- INYECCIÓN
    z = psi(np.array([h1[0], h2_0, h3[0]]))
    X = np.zeros((n, 3)); X[0] = [h1[0], h2_0, h3[0]]
    uprev = np.zeros(2)
    for k in range(n - 1):
        zp = A_use @ z + B_use @ uprev
        zp[:3] = np.maximum(zp[:3], 0); zp[6:] = np.maximum(zp[6:], 0)
        y = np.array([np.sqrt(max(h1[k+1], EPS_PSI)), np.sqrt(max(h3[k+1], EPS_PSI)),
                      h1[k+1], h3[k+1]])
        z = zp + L_use @ (y - C_use @ zp)
        z[:3] = np.maximum(z[:3], 0); z[6:] = np.maximum(z[6:], 0)
        X[k+1] = np.clip(z[:3]**2, 0, 0.75)
        uprev = u[k].copy()
    return X, T, h2r


def tau_empirical_injection(path, A_use, B_use, L_use, C_use,
                             window_s=60.0, floor_m=5e-3):
    """
    Ajusta τ_e SOLO sobre la respuesta libre [0, window_s] y SOLO sobre
    puntos por encima del piso (|e| > floor_m). window_s=60s = 2τ_teórico.
    """
    X, T, h2r = replay_injected(path, A_use, B_use, L_use, C_use)
    e = X[:, 1] - h2r
    m = T <= window_s
    t_fit = T[m]
    e_fit = np.abs(e[m])
    pos = e_fit > floor_m
    if pos.sum() < 5:
        return float('nan'), float('nan'), X, T, h2r
    try:
        slope, _ = np.polyfit(t_fit[pos], np.log(e_fit[pos]), 1)
        tau = -1.0 / slope if slope < 0 else float('nan')
    except Exception:
        tau = float('nan')
    # tiempo a 10 % del offset inyectado
    thr = 0.10 * 0.02
    idx = np.where(e_fit <= thr)[0]
    t10 = float(t_fit[idx[0]]) if len(idx) else float('nan')
    return tau, t10, X, T, h2r


# ---------------------------------------------------------------------------
# P5 ejecución
# ---------------------------------------------------------------------------
print("\n" + "=" * 82)
print(f" P5. VERIFICACIÓN EMPÍRICA (INYECCIÓN +2 cm)  dir: {args.csv_dir}")
print("=" * 82)

tau_emp_mean = float('nan')
t10_mean     = float('nan')

if not os.path.isdir(args.csv_dir):
    print(f"  ⚠ Directorio no existe: {args.csv_dir}")
else:
    logs = sorted(glob.glob(os.path.join(args.csv_dir, 'koopman_run*.csv')), key=run_num)
    if not logs:
        print(f"  ⚠ No hay koopman_run*.csv en {args.csv_dir}")
    else:
        taus, t10s = [], []
        print(f"  Método: inyección ĥ₂(0) = h₂_real(0) + 2 cm, fit en [0, 60 s], |e| > 5 mm\n")
        for p in logs:
            try:
                tau, t10, _, _, _ = tau_empirical_injection(p, A, B, L_use, C)
                if np.isfinite(tau):
                    taus.append(tau)
                    if np.isfinite(t10): t10s.append(t10)
                    print(f"  {os.path.basename(p):<24} τ_emp = {tau:6.1f} s   "
                          f"t(10 %) = {t10:5.1f} s")
                else:
                    print(f"  {os.path.basename(p):<24} τ_emp = n/d (fit fallido)")
            except Exception as e:
                print(f"  {os.path.basename(p):<24} error: {type(e).__name__}: {e}")

        if taus:
            tau_emp_mean = float(np.mean(taus))
            std_t = float(np.std(taus, ddof=1)) if len(taus) > 1 else 0.0
            if t10s:
                t10_mean = float(np.mean(t10s))
            print(f"\n  τ_emp (media ± σ) = {tau_emp_mean:.1f} ± {std_t:.1f} s")
            print(f"  τ_teórico          = {tau_use:.2f} s   "
                  f"(= -Ts / ln ρ(A_c), ρ = {rho_use:.5f})")
            if t10s:
                print(f"  t(10 %) empírico   = {t10_mean:.1f} s   "
                      f"(predicción 2.3·τ = {2.3*tau_use:.1f} s)")
            r = tau_emp_mean / tau_use
            print(f"  ratio τ_emp / τ_teórico = {r:.2f}   "
                  + ("✓ consistente" if 0.7 < r < 1.4
                     else "⚠ revisar (posible contaminación del fit)"))

# ============================================================================
# 8. P6 — Bloques LaTeX listos para pegar
# ============================================================================
print("\n" + "=" * 82)
print(" P6. BLOQUES LATEX — copiar a paper / cartas")
print("=" * 82)

print(f"""
% ------------------------------------------------------------------
% Remark 1 (§3.2.2 del paper) — REEMPLAZAR
% ------------------------------------------------------------------
For the experimental observer, the Lyapunov solution of
$\\mathbf A_c^\\top W\\mathbf A_c - W = -I$ with $\\mathbf A_c = (I-LC)\\mathbf A$
yields $\\lambda_{{\\max}}(W) = {const['lam_max']:.4f}$,
$\\lambda_{{\\min}}(W) = {const['lam_min']:.4f}$,
$\\kappa(W) = {const['kappa']:.4f}$,
$\\gamma = {const['gamma']:.5f} < 1$, and amplification factor
$\\sqrt{{\\kappa(W)}}/(1-\\gamma) = {const['amplif']:.2f}$.

% ------------------------------------------------------------------
% §3.2.1 y §4.10 — REEMPLAZAR valores espectrales
% ------------------------------------------------------------------
With $L$ synthesized in predictor form
$L=\\mathbf AXC^\\top(CXC^\\top+R)^{{-1}}$, the closed-loop error matrix is
$\\mathbf A_c=(I-LC)\\mathbf A$, with spectral radius
$\\rho(\\mathbf A_c) = {rho_use:.5f}$, maximum gain magnitude
$|L|_{{\\max}} = {np.abs(L_use).max():.4f}$, and decay time constant
$\\tau_e = -T_s/\\ln\\rho(\\mathbf A_c) = {tau_use:.2f}\\,\\mathrm{{s}}$.

% ------------------------------------------------------------------
% §4.10 (baseline predictor L=0) — REEMPLAZAR
% ------------------------------------------------------------------
The pure predictor ($L=0$) has $\\rho(\\mathbf A) = {np.abs(eigvals(A)).max():.5f}$
and a time constant $\\tau_\\mathrm{{predictor}} = -T_s/\\ln\\rho(\\mathbf A)
\\approx {(-Ts/np.log(np.abs(eigvals(A)).max())):.0f}\\,\\mathrm{{s}}$, i.e., it cannot
converge within the 800~s horizon.

% ------------------------------------------------------------------
% Corrección de notación (para la carta R2:1 y R3:2)
% ------------------------------------------------------------------
The error dynamics is $e_k = (I-LC)\\,\\mathbf A\\, e_{{k-1}} + \\varepsilon_k$
with $\\varepsilon_k = (I-LC) r_{{k-1}} - L v_k$.
The closed-loop matrix used for the spectral analysis is therefore
$\\mathbf A_c = (I-LC)\\,\\mathbf A$, not $\\mathbf A - LC$.
The previously reported value $\\rho=0.94262$ ($\\tau_e=33.7$~s) corresponded
to $\\mathbf A-LC$; the corrected value is
$\\rho(\\mathbf A_c) = {rho_use:.5f}$ ($\\tau_e = {tau_use:.2f}$~s).
""")

if np.isfinite(tau_emp_mean):
    print(f"""
% ------------------------------------------------------------------
% Para la carta R3:7 — añadir respaldo empírico
% ------------------------------------------------------------------
The empirical decay constant fitted on the five logged runs is
$\\tau_e^\\mathrm{{emp}} = {tau_emp_mean:.1f}\\,\\mathrm{{s}}$, consistent with the
theoretical value $\\tau_e = {tau_use:.2f}\\,\\mathrm{{s}}$ derived from
$\\rho(\\mathbf A_c)$.
""")

print("=" * 82)
print(" FIN")
print("=" * 82)

 P1. VERIFICACIÓN ESPECTRAL — (I-LC)A vs A-LC

  |L|max  = 0.5867
  ρ(A)    = 0.99993

  Matrices candidatas:
  (I-LC)A  [CORRECTA]        ρ = 0.935146   τ_e =      29.83 s   OK
  A-LC     [INCORRECTA]      ρ = 0.942666   τ_e =      33.87 s   OK

  Autovalores (módulo, ordenados):
   (I-LC)A: [0.9351 0.6565 0.4815 0.4093 0.3867 0.3496 0.3496 0.0343]
   A-LC   : [0.9427 0.6776 0.3892 0.3743 0.3743 0.354  0.3026 0.0289]
   Δ máx entre espectros = 9.24e-02  → reemplazar

 P2. REPRODUCCIÓN DEL GAIN VÍA DARE (alpha=1, R=I4)

  ||L_paper - L_predictor||_max = 3.04e-04
  ||L_paper - L_filter   ||_max = 9.79e-02
  → El paper usa la forma PREDICTOR: L = A X Cᵀ (CXCᵀ+R)⁻¹

✓ (I-LC)A ES Schur con el gain del paper.
  → Solo corregir notación + reemplazar ρ, τ_e, γ, κ(W).

 P3. CONSTANTES DE LYAPUNOV  (A_c = (I-LC)A)

  λ_max(W) =    11.3262   (paper viejo: 11.32)
  λ_min(W) =     1.0012   (paper viejo: 1.001)
  κ(W)     =    11.3129   (paper viejo: 11.31)
  γ        =    0.95483   (paper viejo: 0